In [ ]:
# Find and drop false positives, then split into Overture and 3D-GloBFP layers

import arcpy
import os

# ============================================================
# SETTINGS
# ============================================================

city = "CHANGE_ME"

GDB = r"C:\Users\msw2198\Documents\ArcGIS\Projects\MyProject\MyProject.gdb"

# ------------------------------------------------------------
# INPUTS
# ------------------------------------------------------------

OUTPUT_GPKG = (
    rf"E:\buildings\{city}\data"
    rf"\{city}_overture_buildings_gee.gpkg"
)

INPUT_LAYER = f"{city}_overture_buildings_gee"

INPUT_BUILDINGS = os.path.join(
    OUTPUT_GPKG,
    INPUT_LAYER
)

# ------------------------------------------------------------
# OUTPUTS
# ------------------------------------------------------------

SCREENED_BUILDINGS = os.path.join(
    GDB,
    "integrated_buildings_false_positive_screened"
)

GLO = os.path.join(
    GDB,
    "glo_buildings_25d_screened"
)

OV = os.path.join(
    GDB,
    "overture_buildings_25d_screened"
)

# ------------------------------------------------------------
# FIELD NAMES
# ------------------------------------------------------------

YEAR_FIELD = "year"
ALPHA_FIELD = "isolation_forest_outlier"
DW_CLASS_FIELD = "top_class_name"
DW_PROB_FIELD_CANDIDATES = [
    "top_probability",
    "top_class_probability"
]
GOOGLE25D_FIELD = "google25d_max_confidence"
GEOMETRY_CONFIDENCE_FIELD = "geometry_confidence"
GEOMETRY_PROVIDER_FIELD = "geometry_provider"

# ------------------------------------------------------------
# THRESHOLDS
# ------------------------------------------------------------

RECENT_YEAR = 2023
STRONG_BUILT_PROBABILITY = 0.60
GOOGLE25D_LOW_CONFIDENCE = 0.10
GOOGLE25D_OLDER_THAN_YEAR = 2024
GLO_PROVIDER_VALUE = "3D-GloBFP"


# ============================================================
# HELPERS
# ============================================================

def normalize(value):

    if value is None:
        return ""

    return str(value).strip().lower()


def to_float(value):

    if value is None:
        return None

    try:
        return float(value)
    except (TypeError, ValueError):
        return None


def to_bool(value):

    if isinstance(value, bool):
        return value

    value_text = normalize(value)

    return value_text in [
        "true",
        "t",
        "1",
        "yes",
        "y"
    ]


def get_year(value):

    year = to_float(value)

    if year is None:
        return None

    return int(year)


def is_overture(provider):

    return normalize(provider) != normalize(GLO_PROVIDER_VALUE)


def is_recent_overture(year, provider):

    year_value = get_year(year)

    if year_value is None:
        return False

    return (
        is_overture(provider)
        and year_value >= RECENT_YEAR
    )


def has_strong_built_support(dw_class, dw_probability):

    probability = to_float(dw_probability)

    if probability is None:
        probability = 0

    return (
        normalize(dw_class) == "built"
        and probability >= STRONG_BUILT_PROBABILITY
    )


def has_weak_or_non_built_dynamic_world(dw_class, dw_probability):

    probability = to_float(dw_probability)

    if normalize(dw_class) != "built":
        return True

    if probability is None:
        return True

    return probability < STRONG_BUILT_PROBABILITY


def has_low_google25d_support(google25d_confidence):

    confidence = to_float(google25d_confidence)

    if confidence is None:
        return False

    return confidence <= GOOGLE25D_LOW_CONFIDENCE


def is_older_than_google25d_year(year):

    year_value = get_year(year)

    if year_value is None:
        return False

    return year_value < GOOGLE25D_OLDER_THAN_YEAR


def geometry_confidence_is_not_high(geometry_confidence):

    return normalize(geometry_confidence) != "high"


def should_drop_false_positive(row):

    (
        year,
        alpha_outlier,
        dw_class,
        dw_probability,
        google25d_confidence,
        geometry_confidence,
        provider
    ) = row

    # Protective checks are evaluated first.
    if is_recent_overture(year, provider):
        return False

    if has_strong_built_support(dw_class, dw_probability):
        return False

    # Rule Set A: AlphaEarth + Dynamic World + geometry confidence.
    rule_set_a = (
        to_bool(alpha_outlier)
        and has_weak_or_non_built_dynamic_world(
            dw_class,
            dw_probability
        )
        and geometry_confidence_is_not_high(
            geometry_confidence
        )
    )

    # Rule Set B: Google 2.5D + year.
    rule_set_b = (
        has_low_google25d_support(
            google25d_confidence
        )
        and is_older_than_google25d_year(
            year
        )
    )

    return (
        rule_set_a
        or rule_set_b
    )


def delete_by_provider(fc, keep_glo):

    deleted = 0

    with arcpy.da.UpdateCursor(
        fc,
        [GEOMETRY_PROVIDER_FIELD]
    ) as cursor:

        for row in cursor:

            provider = normalize(
                row[0]
            )

            is_glo = (
                provider
                ==
                normalize(GLO_PROVIDER_VALUE)
            )

            if is_glo != keep_glo:

                cursor.deleteRow()

                deleted += 1

    return deleted


# ============================================================
# CHECK INPUTS
# ============================================================

print("=" * 70)
print("FALSE POSITIVE SCREENING")
print("=" * 70)

if not arcpy.Exists(INPUT_BUILDINGS):
    raise FileNotFoundError(
        f"Input buildings layer not found:\n{INPUT_BUILDINGS}"
    )

existing_fields = [
    f.name
    for f in arcpy.ListFields(INPUT_BUILDINGS)
]

field_lookup = {
    field.lower(): field
    for field in existing_fields
}

DW_PROB_FIELD = None

for candidate in DW_PROB_FIELD_CANDIDATES:

    if candidate.lower() in field_lookup:

        DW_PROB_FIELD = field_lookup[candidate.lower()]

        break

required_fields = [
    YEAR_FIELD,
    ALPHA_FIELD,
    DW_CLASS_FIELD,
    GOOGLE25D_FIELD,
    GEOMETRY_CONFIDENCE_FIELD,
    GEOMETRY_PROVIDER_FIELD
]

missing_fields = [
    field
    for field in required_fields
    if field.lower() not in field_lookup
]

if DW_PROB_FIELD is None:
    missing_fields.append(
        " or ".join(DW_PROB_FIELD_CANDIDATES)
    )

if missing_fields:
    raise RuntimeError(
        "Missing required field(s): "
        + ", ".join(missing_fields)
    )

YEAR_FIELD = field_lookup[YEAR_FIELD.lower()]
ALPHA_FIELD = field_lookup[ALPHA_FIELD.lower()]
DW_CLASS_FIELD = field_lookup[DW_CLASS_FIELD.lower()]
GOOGLE25D_FIELD = field_lookup[GOOGLE25D_FIELD.lower()]
GEOMETRY_CONFIDENCE_FIELD = field_lookup[GEOMETRY_CONFIDENCE_FIELD.lower()]
GEOMETRY_PROVIDER_FIELD = field_lookup[GEOMETRY_PROVIDER_FIELD.lower()]

print()
print(f"Dynamic World probability field: {DW_PROB_FIELD}")


# ============================================================
# DELETE EXISTING OUTPUTS
# ============================================================

print()
print("Preparing outputs...")

for fc in [SCREENED_BUILDINGS, GLO, OV]:

    if arcpy.Exists(fc):

        print(f"Deleting existing output: {fc}")

        arcpy.management.Delete(fc)


# ============================================================
# COPY INPUTS
# ============================================================

print()
print("Copying integrated buildings...")

arcpy.management.CopyFeatures(
    INPUT_BUILDINGS,
    SCREENED_BUILDINGS
)

original_count = int(
    arcpy.management.GetCount(
        SCREENED_BUILDINGS
    )[0]
)

print(
    f"Original integrated buildings: "
    f"{original_count:,}"
)


# ============================================================
# DROP FALSE POSITIVES
# ============================================================

print()
print("Dropping conservative false positives...")

false_positive_count = 0

with arcpy.da.UpdateCursor(
    SCREENED_BUILDINGS,
    [
        YEAR_FIELD,
        ALPHA_FIELD,
        DW_CLASS_FIELD,
        DW_PROB_FIELD,
        GOOGLE25D_FIELD,
        GEOMETRY_CONFIDENCE_FIELD,
        GEOMETRY_PROVIDER_FIELD
    ]
) as cursor:

    for row in cursor:

        if should_drop_false_positive(row):

            cursor.deleteRow()

            false_positive_count += 1

screened_count = int(
    arcpy.management.GetCount(
        SCREENED_BUILDINGS
    )[0]
)

print(
    f"False positives removed: "
    f"{false_positive_count:,}"
)

print(
    f"Buildings retained:        "
    f"{screened_count:,}"
)


# ============================================================
# SPLIT INTO 3D-GloBFP AND OVERTURE LAYERS
# ============================================================

print()
print("Splitting screened buildings by geometry provider...")

arcpy.management.CopyFeatures(
    SCREENED_BUILDINGS,
    GLO
)

arcpy.management.CopyFeatures(
    SCREENED_BUILDINGS,
    OV
)

deleted_from_glo = delete_by_provider(
    GLO,
    keep_glo=True
)

deleted_from_ov = delete_by_provider(
    OV,
    keep_glo=False
)

glo_count = int(
    arcpy.management.GetCount(
        GLO
    )[0]
)

ov_count = int(
    arcpy.management.GetCount(
        OV
    )[0]
)


# ============================================================
# FINAL REPORT
# ============================================================

print()
print("=" * 70)
print("FALSE POSITIVE SCREENING COMPLETE")
print("=" * 70)

print(
    f"Original buildings:      "
    f"{original_count:,}"
)

print(
    f"False positives dropped: "
    f"{false_positive_count:,}"
)

print(
    f"Screened buildings:      "
    f"{screened_count:,}"
)

print()

print(
    f"3D-GloBFP buildings:     "
    f"{glo_count:,}"
)

print(
    f"Overture buildings:      "
    f"{ov_count:,}"
)

print()
print("OUTPUTS")
print("-------")
print(f"Screened integrated layer: {SCREENED_BUILDINGS}")
print(f"3D-GloBFP layer:           {GLO}")
print(f"Overture layer:            {OV}")

print()
print("=" * 70)
print("DONE")
print("=" * 70)


In [ ]:
# Remove overlaps with 10% IoU

import arcpy
import os

# ============================================================
# SETTINGS
# ============================================================

GDB = r"C:\Users\msw2198\Documents\ArcGIS\Projects\MyProject\MyProject.gdb"

# ------------------------------------------------------------
# INPUTS
# ------------------------------------------------------------

GLO = os.path.join(
    GDB,
    "glo_buildings_25d_screened"
)

OV = os.path.join(
    GDB,
    "overture_buildings_25d_screened"
)

# ------------------------------------------------------------
# OUTPUTS
# ------------------------------------------------------------

OV_KEEP = os.path.join(
    GDB,
    "buildings_cleaned_method2_ov_25d"
)

GLO_KEEP = os.path.join(
    GDB,
    "buildings_cleaned_method2_glo_25d"
)

# ------------------------------------------------------------
# IoU threshold
# ------------------------------------------------------------

IOU_THRESHOLD = 0.10


# ============================================================
# DELETE EXISTING OUTPUTS
# ============================================================

print("=" * 70)
print("PREPARING OUTPUTS")
print("=" * 70)

for fc in [OV_KEEP, GLO_KEEP]:

    if arcpy.Exists(fc):

        print(f"Deleting existing output: {fc}")

        arcpy.management.Delete(fc)


# ============================================================
# COPY INPUTS
# ============================================================

print()
print("Copying input feature classes...")

arcpy.management.CopyFeatures(
    OV,
    OV_KEEP
)

arcpy.management.CopyFeatures(
    GLO,
    GLO_KEEP
)

print("Overture copied.")
print("Glo copied.")


# ============================================================
# ADD UNIQUE IDS
# ============================================================

def add_unique_id(fc, field_name):

    fields = [
        f.name.lower()
        for f in arcpy.ListFields(fc)
    ]

    if field_name.lower() not in fields:

        arcpy.management.AddField(
            fc,
            field_name,
            "LONG"
        )

    i = 1

    with arcpy.da.UpdateCursor(
        fc,
        [field_name]
    ) as cursor:

        for row in cursor:

            row[0] = i

            cursor.updateRow(row)

            i += 1


print()
print("=" * 70)
print("CREATING UNIQUE IDS")
print("=" * 70)

add_unique_id(
    OV_KEEP,
    "OV_ID"
)

add_unique_id(
    GLO_KEEP,
    "GLO_ID"
)


# ============================================================
# CREATE AREA LOOKUPS
# ============================================================

print()
print("=" * 70)
print("CALCULATING BUILDING AREAS")
print("=" * 70)

ov_area = {}

with arcpy.da.SearchCursor(
    OV_KEEP,
    [
        "OV_ID",
        "SHAPE@AREA"
    ]
) as cursor:

    for ov_id, area in cursor:

        if area is not None and area > 0:

            ov_area[ov_id] = float(area)


glo_area = {}

with arcpy.da.SearchCursor(
    GLO_KEEP,
    [
        "GLO_ID",
        "SHAPE@AREA"
    ]
) as cursor:

    for glo_id, area in cursor:

        if area is not None and area > 0:

            glo_area[glo_id] = float(area)


print(
    f"Overture buildings with valid area: "
    f"{len(ov_area):,}"
)

print(
    f"Glo buildings with valid area: "
    f"{len(glo_area):,}"
)


# ============================================================
# CREATE INTERSECTION
# ============================================================

print()
print("=" * 70)
print("CREATING OVERTURE / GLO INTERSECTIONS")
print("=" * 70)

overlap_table = os.path.join(
    GDB,
    "ov_glo_intersections_method2_tmp"
)

if arcpy.Exists(overlap_table):

    arcpy.management.Delete(
        overlap_table
    )


arcpy.analysis.Intersect(
    [
        OV_KEEP,
        GLO_KEEP
    ],
    overlap_table,
    "ALL",
    output_type="INPUT"
)

print("Intersection complete.")


# ============================================================
# IDENTIFY ID FIELDS
# ============================================================

intersection_fields = [
    f.name
    for f in arcpy.ListFields(
        overlap_table
    )
]

print()
print("Intersection fields:")

for field in intersection_fields:

    print(
        f"  {field}"
    )


field_lookup = {
    f.name.lower(): f.name
    for f in arcpy.ListFields(
        overlap_table
    )
}

ov_id_field = field_lookup.get(
    "ov_id"
)

glo_id_field = field_lookup.get(
    "glo_id"
)


if ov_id_field is None:

    raise RuntimeError(
        "Could not find OV_ID in intersection output."
    )


if glo_id_field is None:

    raise RuntimeError(
        "Could not find GLO_ID in intersection output."
    )


print()
print(
    f"Overture ID field: {ov_id_field}"
)

print(
    f"Glo ID field:      {glo_id_field}"
)


# ============================================================
# EVALUATE IoU AND CONTAINMENT
# ============================================================

print()
print("=" * 70)
print("EVALUATING IoU AND CONTAINMENT")
print("=" * 70)

drop_ov = set()

total_intersections = 0
iou_matches = 0
contained_matches = 0


# ------------------------------------------------------------
# Iterate through all Overture / Glo intersections
# ------------------------------------------------------------

with arcpy.da.SearchCursor(
    overlap_table,
    [
        ov_id_field,
        glo_id_field,
        "SHAPE@AREA"
    ]
) as cursor:

    for row in cursor:

        ov_id = row[0]
        glo_id = row[1]
        intersection_area = row[2]

        # ----------------------------------------------------
        # Validate
        # ----------------------------------------------------

        if ov_id is None:
            continue

        if glo_id is None:
            continue

        if intersection_area is None:
            continue

        if intersection_area <= 0:
            continue

        # ----------------------------------------------------
        # Original polygon areas
        # ----------------------------------------------------

        ov_original_area = ov_area.get(
            ov_id
        )

        glo_original_area = glo_area.get(
            glo_id
        )

        if ov_original_area is None:
            continue

        if glo_original_area is None:
            continue

        if ov_original_area <= 0:
            continue

        if glo_original_area <= 0:
            continue

        total_intersections += 1

        # ====================================================
        # CALCULATE IoU
        # ====================================================

        union_area = (
            ov_original_area
            +
            glo_original_area
            -
            intersection_area
        )

        if union_area <= 0:
            continue

        iou = (
            intersection_area
            /
            union_area
        )

        # ====================================================
        # CONDITION 1:
        #
        # IoU > 10%
        # ====================================================

        iou_match = (
            iou > IOU_THRESHOLD
        )

        # ====================================================
        # CONDITION 2:
        #
        # Overture completely within Glo
        # ====================================================

        area_tolerance = 0.000001

        contained = (
            intersection_area
            >=
            ov_original_area * (
                1 - area_tolerance
            )
        )

        # ====================================================
        # DROP OVERTURE IF EITHER CONDITION IS TRUE
        # ====================================================

        if iou_match:

            drop_ov.add(
                ov_id
            )

            iou_matches += 1

        elif contained:

            drop_ov.add(
                ov_id
            )

            contained_matches += 1


# ============================================================
# RESULTS
# ============================================================

print()
print("=" * 70)
print("OVERLAP RESULTS")
print("=" * 70)

print(
    f"Overture/Glo intersecting pairs: "
    f"{total_intersections:,}"
)

print(
    f"Overture buildings meeting IoU > "
    f"{IOU_THRESHOLD:.0%}: "
    f"{iou_matches:,}"
)

print(
    f"Overture buildings fully within Glo: "
    f"{contained_matches:,}"
)

print(
    f"Unique Overture buildings to remove: "
    f"{len(drop_ov):,}"
)


# ============================================================
# DELETE OVERTURE BUILDINGS
# ============================================================

print()
print("=" * 70)
print("REMOVING OVERTURE BUILDINGS")
print("=" * 70)

deleted_ov = 0

with arcpy.da.UpdateCursor(
    OV_KEEP,
    ["OV_ID"]
) as cursor:

    for row in cursor:

        ov_id = row[0]

        if ov_id in drop_ov:

            cursor.deleteRow()

            deleted_ov += 1


print(
    f"Deleted Overture buildings: "
    f"{deleted_ov:,}"
)

print(
    "Glo buildings deleted: 0"
)


# ============================================================
# CLEAN UP TEMPORARY ID FIELDS
# ============================================================

print()
print("=" * 70)
print("CLEANING UP")
print("=" * 70)


for fc, field_name in [
    (OV_KEEP, "OV_ID"),
    (GLO_KEEP, "GLO_ID")
]:

    existing_fields = [
        f.name
        for f in arcpy.ListFields(fc)
    ]

    if field_name in existing_fields:

        arcpy.management.DeleteField(
            fc,
            field_name
        )


# ============================================================
# DELETE TEMPORARY INTERSECTION
# ============================================================

if arcpy.Exists(
    overlap_table
):

    arcpy.management.Delete(
        overlap_table
    )


# ============================================================
# FINAL COUNTS
# ============================================================

final_ov_count = int(
    arcpy.management.GetCount(
        OV_KEEP
    )[0]
)

final_glo_count = int(
    arcpy.management.GetCount(
        GLO_KEEP
    )[0]
)


# ============================================================
# FINAL REPORT
# ============================================================

print()
print()
print("=" * 70)
print("FINAL RESULTS")
print("=" * 70)

print(
    f"Original Overture buildings: "
    f"{len(ov_area):,}"
)

print(
    f"Original Glo buildings:      "
    f"{len(glo_area):,}"
)

print()

print(
    f"Overture removed:             "
    f"{deleted_ov:,}"
)

print(
    f"Glo removed:                  "
    f"0"
)

print()

print(
    f"FINAL Overture buildings:     "
    f"{final_ov_count:,}"
)

print(
    f"FINAL Glo buildings:          "
    f"{final_glo_count:,}"
)

print()
print("OUTPUTS")
print("-------")

print(
    f"Overture: {OV_KEEP}"
)

print(
    f"Glo:      {GLO_KEEP}"
)

print()
print("=" * 70)
print("DONE")
print("=" * 70)


In [ ]:
# Merge back, remove small geometries, and export

import arcpy
import os

# ============================================================
# INPUT GEODATABASE
# ============================================================

GDB = r"C:\Users\msw2198\Documents\ArcGIS\Projects\MyProject\MyProject.gdb"

# ------------------------------------------------------------
# INPUTS
# ------------------------------------------------------------

GLO = os.path.join(
    GDB,
    "buildings_cleaned_method2_glo_25d"
)

OV = os.path.join(
    GDB,
    "buildings_cleaned_method2_ov_25d"
)

# ------------------------------------------------------------
# OUTPUT GEOPACKAGE
# ------------------------------------------------------------

OUT_GPKG = r"E:\buildings\buildings_six_cities_9_13.gpkg"

OUT_LAYER = "buildings_six_cities_9_13"

OUT_FC = os.path.join(
    OUT_GPKG,
    OUT_LAYER
)

# ------------------------------------------------------------
# SMALL GEOMETRY THRESHOLD
# ------------------------------------------------------------

MIN_AREA_M2 = 16
AREA_FIELD = "area_m2"

# ============================================================
# CHECK INPUTS
# ============================================================

print("=" * 70)
print("MERGING CLEANED BUILDING LAYERS")
print("=" * 70)

if not arcpy.Exists(GLO):
    raise FileNotFoundError(
        f"GloBFP input not found:\n{GLO}"
    )

if not arcpy.Exists(OV):
    raise FileNotFoundError(
        f"Overture input not found:\n{OV}"
    )

# ============================================================
# CHECK / CREATE OUTPUT DIRECTORY
# ============================================================

out_folder = os.path.dirname(OUT_GPKG)

if not os.path.exists(out_folder):
    os.makedirs(out_folder)

# ============================================================
# CREATE GEOPACKAGE
# ============================================================

if not arcpy.Exists(OUT_GPKG):
    print("\nCreating GeoPackage...")
    
    arcpy.management.CreateSQLiteDatabase(
        OUT_GPKG,
        "GEOPACKAGE"
    )

else:
    print("\nGeoPackage already exists.")

# ============================================================
# DELETE EXISTING OUTPUT
# ============================================================

if arcpy.Exists(OUT_FC):
    print("\nDeleting existing output layer...")
    arcpy.management.Delete(OUT_FC)

# ============================================================
# REPORT INPUT COUNTS
# ============================================================

glo_count = int(arcpy.management.GetCount(GLO)[0])
ov_count = int(arcpy.management.GetCount(OV)[0])

print("\nInput feature counts:")
print(f"  GloBFP:    {glo_count:,}")
print(f"  Overture:  {ov_count:,}")
print(f"  Total:     {glo_count + ov_count:,}")

# ============================================================
# MERGE
# ============================================================

print("\nMerging layers...")

arcpy.management.Merge(
    [GLO, OV],
    OUT_FC
)

# ============================================================
# REMOVE SMALL GEOMETRIES
# ============================================================

print()
print("=" * 70)
print("REMOVING SMALL GEOMETRIES")
print("=" * 70)

existing_fields = [
    f.name
    for f in arcpy.ListFields(OUT_FC)
]

if AREA_FIELD not in existing_fields:
    raise RuntimeError(
        f"Missing required area field: {AREA_FIELD}"
    )

small_geometry_count = 0

with arcpy.da.UpdateCursor(
    OUT_FC,
    [AREA_FIELD]
) as cursor:

    for row in cursor:

        area_m2 = row[0]

        if area_m2 is not None and area_m2 < MIN_AREA_M2:

            cursor.deleteRow()

            small_geometry_count += 1

print(
    f"Removed geometries with {AREA_FIELD} < "
    f"{MIN_AREA_M2} m2: "
    f"{small_geometry_count:,}"
)

# ============================================================
# VERIFY OUTPUT
# ============================================================

out_count = int(arcpy.management.GetCount(OUT_FC)[0])
expected_count = glo_count + ov_count - small_geometry_count

print("\n" + "=" * 70)
print("MERGE COMPLETE")
print("=" * 70)

print(f"GloBFP features:       {glo_count:,}")
print(f"Overture features:     {ov_count:,}")
print(f"Merged total:          {glo_count + ov_count:,}")
print(f"Small geometries cut:  {small_geometry_count:,}")
print(f"Expected final total:  {expected_count:,}")
print(f"Output features:       {out_count:,}")

print("\nOutput:")
print(OUT_FC)

if out_count == expected_count:
    print("\nFeature count check: PASSED")
else:
    print("\nWARNING: Output count differs from expected total.")

print("\nDone.")
